# Demo - generating absorber geometries from a target spectrum

This notebook is an **inference-only** demonstration: it restores the trained conditional DDPM written by `MNCC_diffusion_pipeline.ipynb` and shows how candidate cell geometries are generated, re-simulated, ranked and visualised. It does **not** train, create mock data, or write/replace artifacts.

**Prerequisites**

- `src/mnc_inference.py` - the canonical inference module (same definitions as the training pipeline).
- `mnc_artifacts/` produced by the training pipeline (Part 9):
  - `mnc_ddpm.pt` - trained weights, architecture, standardisation and physics constants;
  - `demo_targets.npz` - reference spectra (`A`) and the designs that produced them (`X`);
  - `results.json` - optional evaluation numbers from that run.

**Read-only principle.** The demo is a consumer of verified training artifacts. If a required file is missing, setup fails with an explicit error instead of inventing a replacement.

**Running it.** Keep `src/mnc_inference.py` and `mnc_artifacts/` in the working directory, or set `MNC_MODULE_DIR` / `MNC_ARTIFACT_DIR`. In Colab the artifacts are expected at `/content/drive/MyDrive/mnc_backup/mnc_artifacts` and Drive is mounted automatically. If the module source changes during a session, restart the runtime and re-import it cleanly.


## 0. Environment setup

In [ ]:
# --- Environment setup -------------------------------------------------------
# Imports only: this notebook never writes to the artifact directory.
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 1. Paths, module import and artifacts

One canonical `ARTIFACT_DIR` and one import of `mnc_inference`; later cells reuse this state.

In [ ]:
# --- Paths and module import (exactly once) ---------------------------------
PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "mnc_inference.py").is_file()), Path.cwd())
MODULE_DIR = Path(os.environ.get("MNC_MODULE_DIR", PROJECT_DIR / "src")).expanduser().resolve()

if "MNC_ARTIFACT_DIR" in os.environ:
    ARTIFACT_DIR = Path(os.environ["MNC_ARTIFACT_DIR"]).expanduser().resolve()
elif IN_COLAB:
    ARTIFACT_DIR = Path("/content/drive/MyDrive/mnc_backup/mnc_artifacts")
else:
    ARTIFACT_DIR = (PROJECT_DIR / "mnc_artifacts").resolve()

module_file = MODULE_DIR / "mnc_inference.py"
if not module_file.is_file():
    raise FileNotFoundError(
        f"Canonical inference module missing: {module_file}. "
        "Keep the project layout intact or set MNC_MODULE_DIR."
    )

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import mnc_inference as mnc

if Path(mnc.__file__).resolve() != module_file:
    raise ImportError(
        f"mnc_inference was imported from {mnc.__file__}, not from {module_file}. "
        "Restart the runtime and re-run the notebook."
    )

print("module   :", module_file)
print("artifacts:", ARTIFACT_DIR)


In [ ]:
# --- Validate the artifacts and load the trained model once ------------------
checkpoint_path = ARTIFACT_DIR / mnc.ARTIFACT_FILES["checkpoint"]
demo_path = ARTIFACT_DIR / mnc.ARTIFACT_FILES["demo"]
results_path = ARTIFACT_DIR / mnc.ARTIFACT_FILES["results"]

missing = [p for p in (checkpoint_path, demo_path) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Required artifact(s) missing:\n  " + "\n  ".join(map(str, missing))
        + "\nRun MNCC_diffusion_pipeline.ipynb (Part 9) or point MNC_ARTIFACT_DIR at the trained artifacts."
    )

# mnc.load validates the architecture (D=13, C=81, h=256, nb=5), the 200 diffusion
# steps, the mu/sd standardisation and the physics constants before restoring weights.
meta = mnc.load(ARTIFACT_DIR)

if not {"arch", "tsteps", "physics"} <= set(meta):
    raise RuntimeError("mnc_inference.py is out of date: load() did not return validated metadata.")

ph = meta["physics"]
print("restored architecture:", meta["arch"])
print("diffusion steps      :", meta["tsteps"])
print("physics grid         : %.0f-%.0f Hz (%d points)" % (ph["FREQ"][0], ph["FREQ"][-1], len(ph["FREQ"])))
print("physics constants    : LSCALE=%g  EPS=%g  DEL=%g  W=%g  T=%g  TW=%g  HIN=%g"
      % (ph["LSCALE"], ph["EPS"], ph["DEL"], ph["W"], ph["T"], ph["TW"], ph["HIN"]))
print("training metadata    :", json.dumps(meta["train_meta"], indent=1, default=str))

if results_path.is_file():
    with open(results_path) as f:
        results = json.load(f)
    print("results.json keys    :", ", ".join(results))
else:
    print("results.json         : not present (optional for inference)")


## 2. Validate the reference dataset

In [ ]:
# --- Validate the reference dataset -----------------------------------------
with np.load(demo_path) as data:
    A = data["A"].copy()
    X = data["X"].copy()

assert A.ndim == 2 and A.shape[1] == mnc.NF, f"A must have shape (n, {mnc.NF}); got {A.shape}"
assert X.shape == (A.shape[0], mnc.D), f"X must have shape ({A.shape[0]}, {mnc.D}); got {X.shape}"
assert np.isfinite(A).all(), "A contains NaN or infinite values"
assert np.isfinite(X).all(), "X contains NaN or infinite values"
assert np.all((A >= 0) & (A <= 1)), "absorption values in A must lie in [0, 1]"

N_REF = A.shape[0]
assert np.allclose(mnc.reference_spectrum(0), A[0]), "module and notebook read different demo spectra"
assert np.allclose(mnc.reference_design(0), X[0]), "module and notebook read different demo designs"

print("reference spectra :", A.shape, "| valid indices 0 ..", N_REF - 1)
print("reference designs :", X.shape)


## 3. Generate candidates for a reference target spectrum

64 candidates are sampled, re-simulated with the physics model and the best four are shown. The dashed blue line is the acoustic path from the tube mouth (top) to the rigid end.

In [ ]:
GUIDANCE = 1.0        # classifier-free guidance (1.0 = plain conditional sampling)
N_CANDIDATES = 64     # sampled first, then ranked by the physics model
SEED = 0              # fixes the sampling noise so the result is reproducible
INDEX = 5             # reference spectrum to reproduce

assert 0 <= INDEX < N_REF, f"INDEX must lie in [0, {N_REF - 1}]"

target = mnc.reference_spectrum(INDEX)
result = mnc.generate_designs(target, n=4, scale=GUIDANCE, n_candidates=N_CANDIDATES, seed=SEED)
mnc.gallery(result, 4); plt.show()

print(f"reference index {INDEX} | guidance {GUIDANCE} | candidates {N_CANDIDATES} | seed {SEED}")
print("spectrum RMSE of the four best candidates:", np.round(result.rmse, 3))


## 4. Design card of the best candidate

In [ ]:
fig, metrics = mnc.design_card(result.x[0], target=target, title="Best generated design"); plt.show()
print(json.dumps(metrics, indent=1))


## 5. The original design that produced this spectrum

The reference spectrum comes from a held-out design; the generated geometries need not resemble it, because different geometries can give nearly the same spectrum.

In [ ]:
fig, _ = mnc.design_card(mnc.reference_design(INDEX), target=target, title="Original design (source of the target)"); plt.show()


## 6. Interactive exploration

Pick a reference spectrum, slide it along frequency (the documented reliable range is about +-6 %, so the slider is limited to it), and choose the guidance weight, candidate count and seed. Each result reports the settings used and the achieved RMSE.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

w_idx = widgets.IntSlider(value=5, min=0, max=N_REF - 1, description="reference")
w_shift = widgets.FloatSlider(value=0, min=-6, max=6, step=0.5, description="shift [%]")
w_scale = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.5, description="guidance")
w_cand = widgets.Dropdown(options=[16, 64, 256], value=64, description="candidates")
w_seed = widgets.IntText(value=0, description="seed")
w_btn = widgets.Button(description="Generate", button_style="primary")
w_out = widgets.Output()

def run(_):
    with w_out:
        clear_output(wait=True)
        tgt = mnc.shift_spectrum(mnc.reference_spectrum(w_idx.value), w_shift.value / 100)
        res = mnc.generate_designs(tgt, n=4, scale=w_scale.value, n_candidates=w_cand.value,
                                   seed=w_seed.value)
        mnc.gallery(res, 4); plt.show()
        mnc.design_card(res.x[0], target=tgt, title="Best candidate"); plt.show()
        print(f"reference index {w_idx.value} | shift {w_shift.value:+.1f} % | guidance {w_scale.value} "
              f"| candidates {w_cand.value} | seed {w_seed.value}")
        print("spectrum RMSE of the four best candidates:", np.round(res.rmse, 3))

w_btn.on_click(run)
display(widgets.VBox([w_idx, w_shift, w_scale, w_cand, w_seed, w_btn, w_out]))


## 7. Own target spectrum

A target is an array of 81 absorption values on `mnc.FREQ` (200-1000 Hz, 10 Hz steps), values in [0, 1]. The cell validates shape, finiteness and range before inference. Targets that look like real spectra of this cell family work best; hand-drawn shapes that the family cannot produce are matched only approximately.

In [ ]:
print("frequency grid:", mnc.FREQ[:3], "...", mnc.FREQ[-1], "| points:", mnc.NF)

# Example: a reference spectrum with its overall level lowered by 10 %.
# For measurements: my_target = np.interp(mnc.FREQ, f_measured_Hz, alpha_measured)
my_target = np.clip(0.9 * mnc.reference_spectrum(0), 0, 1)

my_target = np.asarray(my_target, dtype=np.float32)
assert my_target.shape == (mnc.NF,), f"Target must contain exactly {mnc.NF} values."
assert np.isfinite(my_target).all(), "Target contains NaN or infinite values."
assert np.all((my_target >= 0) & (my_target <= 1)), "Absorption values must be between 0 and 1."

res = mnc.generate_designs(my_target, n=4, scale=1.0, n_candidates=128, seed=1)
mnc.gallery(res, 4); plt.show()
print("spectrum RMSE of the four best candidates:", np.round(res.rmse, 3))
print("Simulator-based ranking only: a poor match may mean the target is unreachable by this cell family.")


## Scope and limits

Candidates are ranked by the analytical forward model (transmission line with thermoviscous losses) - not by FEM and not by experiment. A valid 81-point target does not guarantee a geometrically matching design: many targets are physically unreachable by this single-cell, six-channel family. Use the reported RMSE and the re-simulated spectra as the honest measure of match, and validate final designs independently before manufacturing.
